# Hugging Face — Fine-Tuning & Embeddings for Data Engineering NLP Tasks

[![HuggingFace](https://img.shields.io/badge/🤗-Transformers-yellow)](https://huggingface.co/transformers)
[![Python](https://img.shields.io/badge/python-3.11-blue)]()
[![License](https://img.shields.io/badge/license-MIT-lightgrey)]()

This notebook demonstrates two complementary HuggingFace workflows applied to **data engineering use cases**:

**Part 1 — Embeddings with `sentence-transformers`**  
Generate semantic embeddings for data schema elements and column descriptions.  
Practical use: schema matching, data catalogue search, data lineage similarity detection.

**Part 2 — Fine-Tuning a Small Classifier with `transformers` + `Trainer`**  
Fine-tune `distilbert-base-uncased` on a data quality classification task:  
classifying pipeline log messages as `[normal, warning, error, anomaly]`.  
Practical use: automated triage of pipeline alerts — feeds into the AI-Driven Data Quality Platform.

---

**Runtime:** Google Colab (free T4 GPU) or local CPU (Part 1 runs fine on CPU; Part 2 benefits from GPU)  
**Model sizes:** `all-MiniLM-L6-v2` (22M params) and `distilbert-base-uncased` (66M params) — both small enough to run without enterprise GPU budget.

## Setup

In [ ]:
# Install dependencies
# Uncomment the pip installs when running in Colab or a fresh environment

# !pip install transformers datasets sentence-transformers scikit-learn \
#              accelerate evaluate torch --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from pathlib import Path

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device    : {device}")

---
## Part 1 — Semantic Embeddings for Data Schema Matching

### Use Case
When migrating from Hadoop/Hive to Azure (as done at ALDI DX), a major challenge is **schema mapping** — identifying which legacy columns correspond to which new columns when names differ.  
Semantic embeddings let us match `cust_id` → `customer_identifier` → `client_number` by meaning, not just string similarity.

### Model: `sentence-transformers/all-MiniLM-L6-v2`
- 22M parameters, runs fast on CPU
- Produces 384-dimensional sentence embeddings
- Optimised for semantic similarity tasks

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load the model — downloads ~90MB on first run, cached thereafter
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Embedding model loaded. Output dimensions: {embedding_model.get_sentence_embedding_dimension()}")

In [ ]:
# ── Simulated schema migration scenario ───────────────────────────────────────
# Legacy Hive schema (source) — column names + descriptions from the old system
legacy_columns = [
    {"name": "cust_id",        "description": "Unique identifier for the customer in the legacy CRM"},
    {"name": "ord_amt",         "description": "Total monetary value of the order in local currency"},
    {"name": "txn_ts",          "description": "Timestamp when the payment transaction was processed"},
    {"name": "prod_sku",        "description": "Stock keeping unit code for the product ordered"},
    {"name": "dlv_status",      "description": "Current delivery and fulfilment status of the order"},
    {"name": "geo_region",      "description": "Geographic sales region for reporting aggregation"},
    {"name": "is_premium_flg",  "description": "Boolean flag indicating premium tier customer"},
]

# New Azure Databricks schema (target) — dbt Silver/Gold naming convention
new_columns = [
    {"name": "customer_id",            "description": "Primary key for customer dimension"},
    {"name": "order_amount_eur",        "description": "Order value in EUR, rounded to 2 decimal places"},
    {"name": "transaction_timestamp",   "description": "UTC timestamp of payment event"},
    {"name": "product_sku",             "description": "Product stock-keeping unit identifier"},
    {"name": "order_status",            "description": "Normalised order lifecycle status"},
    {"name": "country_code",            "description": "ISO 2-letter country code for the sale"},
    {"name": "is_premium_customer",     "description": "True if customer is in premium segment"},
]

# Embed using column_name + description for richer semantic signal
legacy_texts = [f"{c['name']}: {c['description']}" for c in legacy_columns]
new_texts    = [f"{c['name']}: {c['description']}" for c in new_columns]

print("Encoding legacy schema...")
legacy_embeddings = embedding_model.encode(legacy_texts, convert_to_tensor=True, show_progress_bar=True)

print("Encoding new schema...")
new_embeddings = embedding_model.encode(new_texts, convert_to_tensor=True, show_progress_bar=True)

print(f"\nEmbedding shape per column: {legacy_embeddings[0].shape}")

In [ ]:
# ── Cosine similarity matrix — legacy vs new columns ─────────────────────────
cosine_scores = util.cos_sim(legacy_embeddings, new_embeddings).numpy()

# Build match results
results = []
for i, legacy_col in enumerate(legacy_columns):
    best_match_idx = np.argmax(cosine_scores[i])
    best_score     = cosine_scores[i][best_match_idx]
    results.append({
        "legacy_column":  legacy_col["name"],
        "matched_column": new_columns[best_match_idx]["name"],
        "similarity":     round(float(best_score), 4),
        "confidence":     "HIGH" if best_score > 0.85 else "MEDIUM" if best_score > 0.70 else "LOW",
    })

df_results = pd.DataFrame(results)
print("\n── Schema Matching Results ───────────────────────────────────────")
print(df_results.to_string(index=False))
print()
high_confidence = (df_results['confidence'] == 'HIGH').sum()
print(f"High-confidence matches: {high_confidence}/{len(legacy_columns)}")

In [ ]:
# ── Visualise the similarity heatmap ─────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    cosine_scores,
    xticklabels=[c['name'] for c in new_columns],
    yticklabels=[c['name'] for c in legacy_columns],
    annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Schema Column Similarity Matrix (Cosine Similarity)', fontsize=13, pad=15)
ax.set_xlabel('New Schema (Azure Databricks)', labelpad=10)
ax.set_ylabel('Legacy Schema (Hive)', labelpad=10)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('schema_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: schema_similarity_heatmap.png")

In [ ]:
# ── Bonus: semantic search over a data catalogue ──────────────────────────────
# Simulates searching your Purview / Unity Catalog catalogue by natural language

catalogue_entries = [
    "customer_id: Unique customer identifier across all systems",
    "order_amount_eur: Total order value net of discounts in EUR",
    "transaction_timestamp: UTC timestamp of payment confirmation",
    "days_since_last_order: Recency metric — days since customer last completed an order",
    "lifetime_revenue_eur: Sum of all completed order values for the customer lifetime",
    "is_high_risk: True if customer has any chargeback in their transaction history",
    "chargeback_count: Number of payment chargebacks raised by this customer",
    "fraud_velocity_1m: Number of transactions in the last 1 minute — used in fraud models",
]

catalogue_embeddings = embedding_model.encode(catalogue_entries, convert_to_tensor=True)

def catalogue_search(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, catalogue_embeddings)[0]
    top_results = torch.topk(scores, k=top_k)
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    for score, idx in zip(top_results.values, top_results.indices):
        print(f"  [{score:.3f}] {catalogue_entries[idx.item()]}")

catalogue_search("how long since the customer last bought something")
catalogue_search("suspicious payment behaviour for fraud detection")
catalogue_search("total money spent by customer over their lifetime")

---
## Part 2 — Fine-Tuning DistilBERT for Pipeline Log Classification

### Use Case
In production, data pipelines generate thousands of log messages per hour. At ALDI DX, the AI-Driven Data Quality Platform needed to automatically classify pipeline log messages to trigger the right remediation action — without a human reading every line.

This section fine-tunes `distilbert-base-uncased` (66M params, 40% smaller than BERT) on a log classification task:

| Label | Meaning | Example |
|-------|---------|--------|
| `0 — normal` | Routine log, no action needed | "Pipeline completed successfully. 1,240 rows written." |
| `1 — warning` | Degraded but not failed | "Checkpoint write took 45s — approaching 60s SLA threshold." |
| `2 — error` | Pipeline failure requiring alert | "NullPointerException in Silver transformation. Job aborted." |
| `3 — anomaly` | Statistical anomaly, possible data quality issue | "Row count 0 — expected ~50,000 based on last 7 days average." |

### Why fine-tune instead of zero-shot?
Log messages contain domain-specific jargon (`Auto Loader`, `Delta Live Tables`, `Z-Order`, `SLA breach`) that a general-purpose classifier misinterprets. Fine-tuning on even a small labelled dataset (200–500 examples) dramatically outperforms zero-shot on domain-specific text.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import evaluate
from sklearn.model_selection import train_test_split

# Label mapping
ID2LABEL = {0: "normal", 1: "warning", 2: "error", 3: "anomaly"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

MODEL_NAME = "distilbert-base-uncased"
print(f"Model: {MODEL_NAME}")
print(f"Labels: {ID2LABEL}")

In [ ]:
# ── Training data — realistic pipeline log messages ───────────────────────────
# In production: replace with labelled logs exported from Azure Monitor
# or Databricks logging tables. 200+ examples per class recommended.

raw_data = [
    # ── NORMAL (label 0) ────────────────────────────────────────────────────
    ("Pipeline completed successfully. 12,450 rows written to Silver layer.", 0),
    ("Auto Loader checkpoint updated. 3 new files ingested from ADLS Gen2.", 0),
    ("Delta Live Tables run finished. Bronze → Silver → Gold in 4m 12s.", 0),
    ("Databricks job cluster terminated after autotermination (idle 30 min).", 0),
    ("Unity Catalog lineage scan completed. 47 tables catalogued.", 0),
    ("dbt run completed. 12 models passed, 0 warnings, 0 errors.", 0),
    ("Kafka consumer lag at 0. All partitions fully consumed.", 0),
    ("Incremental merge completed. 2,301 rows upserted into fct_orders.", 0),
    ("Z-Order optimisation completed on gold.fct_orders. 1.2GB compacted.", 0),
    ("CI/CD pipeline passed all stages. Deployment to prod approved.", 0),
    ("Airflow DAG lakehouse_daily succeeded. Duration: 18m 42s.", 0),
    ("MLflow run logged. Model metrics: accuracy=0.94, AUC=0.97.", 0),
    ("Schema validation passed. All 47 columns match expected types.", 0),
    ("Purview scan completed. 0 new PII violations detected.", 0),
    ("Feature store publish complete. 45,200 customer features updated.", 0),

    # ── WARNING (label 1) ────────────────────────────────────────────────────
    ("Checkpoint write took 52s — approaching 60s SLA threshold.", 1),
    ("Kafka consumer lag at 15,000 messages. Throughput degraded.", 1),
    ("ADF pipeline retry #2. Transient timeout on source connection.", 1),
    ("Silver table stg_orders has 3 rows failing not_null test. Flagged for review.", 1),
    ("Cluster autoscaling at max workers (16/16). Consider increasing limit.", 1),
    ("dbt model mart_customer_lifetime took 8m 40s — exceeds 5min target.", 1),
    ("MLflow model drift score 0.12 — above 0.10 warning threshold.", 1),
    ("Auto Loader found 2 malformed JSON files in Bronze. Quarantined.", 1),
    ("Databricks SQL warehouse queue depth at 8. Response time degraded.", 1),
    ("Incremental run skipped 1,200 rows due to schema mismatch in source.", 1),
    ("Azure Monitor alert: pipeline P95 latency exceeded 45 minutes.", 1),
    ("Unity Catalog: 5 tables missing column-level descriptions. Governance gap.", 1),
    ("Feature store freshness at 4m 30s — above 2-minute SLA target.", 1),
    ("Dagster asset mart_customer_lifetime is 2 hours stale.", 1),
    ("Row count in stg_transactions decreased 18% vs 7-day average.", 1),

    # ── ERROR (label 2) ─────────────────────────────────────────────────────
    ("NullPointerException in Silver transformation. Job aborted after 3 retries.", 2),
    ("ADF pipeline 'ingest_orders' failed. Error: Connection timeout after 120s.", 2),
    ("Delta Live Tables pipeline failed. Schema enforcement rejected 4,200 rows.", 2),
    ("Databricks job cluster failed to start. Out of capacity in westeurope.", 2),
    ("dbt test failed: unique constraint violated on fct_orders.order_id.", 2),
    ("Kafka broker unreachable. Producer could not connect after 5 attempts.", 2),
    ("ADLS Gen2 write failed. StorageException: AuthorizationPermissionMismatch.", 2),
    ("PySpark job OOM error. Executor lost on 6 of 8 nodes.", 2),
    ("Unity Catalog permission denied. Service principal missing DATA_READ on gold schema.", 2),
    ("MLflow model registration failed. Registry server returned 503.", 2),
    ("Terraform apply failed. Resource 'azurerm_databricks_workspace' already exists.", 2),
    ("CI/CD deployment failed. dbt run error: model stg_orders returned non-zero exit code.", 2),
    ("Airflow DAG lakehouse_daily failed at task ingest_customers. Upstream missing.", 2),
    ("FastAPI feature store endpoint returned 500. Redis connection refused.", 2),
    ("Purview scan failed on ADLS Bronze container. Managed identity not authorised.", 2),

    # ── ANOMALY (label 3) ────────────────────────────────────────────────────
    ("Row count 0 for stg_orders — expected ~50,000 based on 7-day rolling average.", 3),
    ("Transaction amount distribution shift detected. Mean EUR 42 vs expected EUR 18.", 3),
    ("Duplicate rate in Bronze raw_orders jumped to 34% — baseline is <0.5%.", 3),
    ("NULL rate in customer_id column: 8.2% — previously 0.01% over 90 days.", 3),
    ("feature store: avg_spend_1h is 0 for 99% of customers — possible upstream issue.", 3),
    ("Schema drift detected in raw_transactions: column 'payment_method' dropped.", 3),
    ("Silver stg_customers row count 142 — expected 45,000+ based on CRM snapshot size.", 3),
    ("LLM quality agent flagged 1,200 rows with anomalous order_amount_eur values >10x IQR.", 3),
    ("Fraud model scored 94% of transactions as high risk today vs 2% baseline.", 3),
    ("MLflow data drift score 0.67 on customer_segment feature — distribution shift.", 3),
    ("Kafka topic 'orders' produced 0 messages in last 10 minutes during peak hours.", 3),
    ("gold.mart_customer_lifetime: lifetime_revenue_eur is negative for 3,200 customers.", 3),
    ("Auto Loader batch size 10x larger than previous 30-day average. Possible backfill.", 3),
    ("CDC job detected 40,000 deletes in source — 100x daily average. Possible truncation.", 3),
    ("Unity Catalog lineage scan shows 0 tables with lineage — metadata may have been reset.", 3),
]

texts  = [d[0] for d in raw_data]
labels = [d[1] for d in raw_data]

print(f"Total examples: {len(raw_data)}")
for label_id, label_name in ID2LABEL.items():
    count = labels.count(label_id)
    print(f"  {label_name:8s} (label {label_id}): {count} examples")

In [ ]:
# ── Train / validation split ──────────────────────────────────────────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Train: {len(train_texts)} | Validation: {len(val_texts)}")

# ── Tokenisation ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,   # Log messages are short; 128 tokens is sufficient
        padding=False,    # DataCollatorWithPadding handles dynamic padding
    )

# Build HuggingFace Dataset objects
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels})
val_dataset   = Dataset.from_dict({'text': val_texts,   'labels': val_labels})

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
val_dataset   = val_dataset.map(tokenize,   batched=True, remove_columns=['text'])

print(f"Tokenised train features: {train_dataset.features}")

# Dynamic padding collator — more efficient than padding to max_length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# ── Load pre-trained DistilBERT with classification head ──────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average='weighted')
    return {'accuracy': acc['accuracy'], 'f1_weighted': f1['f1']}

In [ ]:
# ── Training arguments ────────────────────────────────────────────────────────
OUTPUT_DIR = Path('models/pipeline-log-classifier')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # Training schedule
    num_train_epochs=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',

    # Batch sizes — reduce if OOM on CPU
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,   # Effective batch size = 16

    # Evaluation and saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,

    # Logging
    logging_steps=10,
    report_to='none',     # Set to 'mlflow' to log to MLflow

    # Hardware
    fp16=torch.cuda.is_available(),   # Mixed precision on GPU only
    dataloader_num_workers=0,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer initialised. Starting fine-tuning...")

In [ ]:
# ── Fine-tune ─────────────────────────────────────────────────────────────────
# On CPU: ~3-5 minutes for 60 examples × 8 epochs
# On T4 GPU (Colab): ~30-45 seconds

train_result = trainer.train()

print(f"\n── Training complete ──────────────────")
print(f"Runtime      : {train_result.metrics['train_runtime']:.1f}s")
print(f"Samples/sec  : {train_result.metrics['train_samples_per_second']:.1f}")
print(f"Train loss   : {train_result.metrics['train_loss']:.4f}")

In [ ]:
# ── Evaluate on validation set ────────────────────────────────────────────────
eval_results = trainer.evaluate()

print("\n── Validation Metrics ────────────────")
print(f"Loss         : {eval_results['eval_loss']:.4f}")
print(f"Accuracy     : {eval_results['eval_accuracy']:.4f}")
print(f"F1 (weighted): {eval_results['eval_f1_weighted']:.4f}")

# Save the fine-tuned model
trainer.save_model(str(OUTPUT_DIR / 'final'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'final'))
print(f"\nModel saved to: {OUTPUT_DIR}/final")

In [ ]:
# ── Per-class evaluation ──────────────────────────────────────────────────────
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("\n── Per-Class Classification Report ──────────────────────────────")
print(classification_report(
    true_labels, pred_labels,
    target_names=[ID2LABEL[i] for i in range(NUM_LABELS)]
))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, pred_labels)
label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_names, yticklabels=label_names, ax=ax
)
ax.set_title('Confusion Matrix — Pipeline Log Classifier', fontsize=12, pad=12)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# ── Inference — use the fine-tuned model in production ───────────────────────
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    'text-classification',
    model=str(OUTPUT_DIR / 'final'),
    tokenizer=str(OUTPUT_DIR / 'final'),
    device=0 if torch.cuda.is_available() else -1,
)

# Test with unseen log messages
test_logs = [
    "Pipeline completed. 8,900 rows loaded into gold.fct_orders.",
    "Kafka consumer lag reached 42,000. Processing throughput is degraded.",
    "PySpark executor failed with OutOfMemoryError. Job terminated.",
    "Row count in stg_customers is 12 — expected 45,000 based on historical average.",
    "Auto Loader ingested 5 new files. Checkpoint updated successfully.",
]

print("── Inference on unseen log messages ──────────────────────────────")
for log_msg in test_logs:
    result = classifier(log_msg)[0]
    label  = result['label']
    score  = result['score']
    icon   = {'normal': '✅', 'warning': '⚠️', 'error': '❌', 'anomaly': '🔍'}.get(label, '?')
    print(f"{icon} [{label:8s} {score:.2%}] {log_msg[:70]}..." if len(log_msg) > 70 else f"{icon} [{label:8s} {score:.2%}] {log_msg}")

In [ ]:
# ── MLflow integration (optional — uncomment to log to MLflow) ───────────────
# In production, this would log to the Databricks-hosted MLflow tracking server.

# import mlflow
# import mlflow.transformers
#
# MLFLOW_TRACKING_URI = "databricks"   # Or: "http://localhost:5000" for local
# mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
# mlflow.set_experiment("/data-quality/pipeline-log-classifier")
#
# with mlflow.start_run(run_name="distilbert-log-classifier-v1"):
#     mlflow.log_params({
#         "model_name": MODEL_NAME,
#         "num_epochs": training_args.num_train_epochs,
#         "learning_rate": training_args.learning_rate,
#         "train_examples": len(train_texts),
#         "val_examples": len(val_texts),
#     })
#     mlflow.log_metrics({
#         "val_accuracy":     eval_results['eval_accuracy'],
#         "val_f1_weighted":  eval_results['eval_f1_weighted'],
#         "val_loss":         eval_results['eval_loss'],
#     })
#     mlflow.transformers.log_model(
#         transformers_model={'model': model, 'tokenizer': tokenizer},
#         artifact_path="pipeline-log-classifier",
#         task="text-classification",
#     )
#     print("Model logged to MLflow.")

print("MLflow block ready — uncomment to log to Databricks MLflow tracking server.")

---
## Summary

| | Part 1 — Embeddings | Part 2 — Fine-Tuning |
|---|---|---|
| **Model** | `all-MiniLM-L6-v2` | `distilbert-base-uncased` |
| **Parameters** | 22M | 66M |
| **Task** | Schema matching, catalogue search | Pipeline log classification |
| **Use case** | Hive → Azure migration, Unity Catalog search | AI-Driven Data Quality Platform alert triage |
| **Library** | `sentence-transformers` | `transformers` + `Trainer` |
| **Runtime** | CPU, <1 min | GPU preferred, ~1 min on T4 |
| **Production path** | FastAPI endpoint + ChromaDB/Pinecone | MLflow model registry → Databricks serving |

### Next Steps
- **Scale training data** — export real log messages from Azure Monitor + label with the existing rule-based system as weak supervision
- **MLflow registration** — uncomment the MLflow block above, register the model, and serve it via Databricks Model Serving
- **Connect to the Data Quality Platform** — replace the rule-based anomaly detector in `LangChain + DLT` pipeline with this fine-tuned classifier for domain-specific accuracy